# 全球 AI 算力全产业链指数 (GAC-Index)

本 Notebook 使用 yfinance 抓取真实历史数据，按给定权重构建指数 OHLC（开高低收），并计算全组合每日成交金额（Turnover），最终用 Plotly 绘制交互式 K 线图。

## 核心方法
1. **归一化**：把每个成分股在回测起始日的收盘价标准化到 100，再乘以权重。
2. **成交金额**：逐股计算 每日成交股数 × 当日收盘价，并横向汇总。

In [ ]:
# 如未安装依赖，请先执行（取消注释）
# !pip install yfinance pandas plotly

import yfinance as yf
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

portfolio_weights = {
    'NVDA': 0.25,
    'TSM': 0.10,
    'AMD': 0.05,
    'AVGO': 0.10,
    'MRVL': 0.05,
    'VRT': 0.10,
    'CEG': 0.10,
    'DELL': 0.05,
    'MSFT': 0.067,
    'GOOGL': 0.067,
    'AMZN': 0.066
}

start_date = '2023-01-01'
end_date = '2026-04-20'

weight_sum = sum(portfolio_weights.values())
if abs(weight_sum - 1.0) > 1e-8:
    raise ValueError(f'权重之和必须为 1.0，当前为 {weight_sum:.6f}')

tickers = list(portfolio_weights.keys())
print(f'正在下载 {len(tickers)} 只成分股数据：{tickers}')

In [ ]:
data = yf.download(
    tickers=tickers,
    start=start_date,
    end=end_date,
    auto_adjust=False,
    progress=False
)

if data.empty:
    raise ValueError('未下载到数据，请检查网络、日期范围或 ticker 是否有效。')

index_df = pd.DataFrame(index=data.index)
index_df['Open'] = 0.0
index_df['High'] = 0.0
index_df['Low'] = 0.0
index_df['Close'] = 0.0
index_df['Turnover'] = 0.0

for ticker, weight in portfolio_weights.items():
    stock_open = data['Open'][ticker].ffill()
    stock_high = data['High'][ticker].ffill()
    stock_low = data['Low'][ticker].ffill()
    stock_close = data['Close'][ticker].ffill()
    stock_volume = data['Volume'][ticker].fillna(0.0)

    base_candidates = stock_close.dropna()
    if base_candidates.empty:
        raise ValueError(f'{ticker} 在所选区间无有效收盘价，无法构建指数。')

    base_price = float(base_candidates.iloc[0])
    if base_price <= 0:
        raise ValueError(f'{ticker} 基准收盘价异常：{base_price}')

    index_df['Open'] += (stock_open / base_price) * 100 * weight
    index_df['High'] += (stock_high / base_price) * 100 * weight
    index_df['Low'] += (stock_low / base_price) * 100 * weight
    index_df['Close'] += (stock_close / base_price) * 100 * weight

    stock_turnover = stock_volume * stock_close
    index_df['Turnover'] += stock_turnover.fillna(0.0)

index_df = index_df.dropna(subset=['Open', 'High', 'Low', 'Close'])

print(f'指数数据行数: {len(index_df)}')
index_df.head()

In [ ]:
fig = make_subplots(
    rows=2, cols=1,
    shared_xaxes=True,
    vertical_spacing=0.03,
    row_heights=[0.7, 0.3],
    specs=[[{'secondary_y': False}], [{'secondary_y': False}]]
)

fig.add_trace(
    go.Candlestick(
        x=index_df.index,
        open=index_df['Open'],
        high=index_df['High'],
        low=index_df['Low'],
        close=index_df['Close'],
        name='GAC-Index K线',
        increasing_line_color='green',
        decreasing_line_color='red'
    ),
    row=1, col=1
)

colors = [
    'green' if c >= o else 'red'
    for c, o in zip(index_df['Close'], index_df['Open'])
]

fig.add_trace(
    go.Bar(
        x=index_df.index,
        y=index_df['Turnover'] / 1e9,
        name='成交金额 (十亿美元)',
        marker_color=colors,
        opacity=0.8
    ),
    row=2, col=1
)

fig.update_layout(
    title='GAC-Index 全球 AI 算力指数 (2023-2026)',
    yaxis_title='指数点位 (基准=100)',
    yaxis2_title='成交金额 (Billion USD)',
    xaxis_rangeslider_visible=False,
    template='plotly_dark',
    height=850
)

fig.show()

In [ ]:
# 可选：导出指数结果用于后续量化分析
# index_df.to_csv('gac_index_ohlc_turnover.csv', encoding='utf-8-sig')
index_df.tail()